# Clase 180 — Tests no paramétricos: Mann-Whitney, Wilcoxon, Kruskal-Wallis

Cuando los datos son asimétricos, ordinales o tienen outliers, los tests basados en **rangos** son preferibles. Vemos Mann-Whitney U (≈ Welch), Wilcoxon signed-rank (≈ pareado) y Kruskal-Wallis (≈ ANOVA), con effect size y post-hoc corregido.

Requiere: `numpy`, `scipy`, `statsmodels`, `matplotlib`.

## 1. Mann-Whitney U + rank-biserial

`H₀`: `P(X > Y) = 0.5`. Trabaja sobre rangos, no sobre medias. El effect size es la rank-biserial `r = 1 - 2U/(n₁·n₂)`.

In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

a = rng.lognormal(1.0, 0.5, 80)     # asimétricos: el t-test es dudoso
b = rng.lognormal(1.25, 0.5, 90)
U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
r_rb = 1 - 2 * U / (len(a) * len(b))
print(f"Mann-Whitney U={U:.0f}  p={p:.4f}")
print(f"rank-biserial r = {r_rb:.3f}")
print(f"medianas: a={np.median(a):.2f}  b={np.median(b):.2f}")
assert p < 0.05

## 2. Wilcoxon signed-rank (pareado)

Análogo no paramétrico del `ttest_rel`. Asume **simetría** de las diferencias, que verificamos con su asimetría.

In [ ]:
antes = rng.normal(140, 12, 30)
despues = antes - rng.normal(5, 3, 30)
w = stats.wilcoxon(antes, despues)
dif = antes - despues
print(f"Wilcoxon signed-rank: stat={w.statistic:.1f}  p={w.pvalue:.2e}")
print(f"asimetría de las diferencias = {stats.skew(dif):.2f}  (≈0 => simetría OK)")
assert w.pvalue < 0.01

plt.figure(figsize=(6, 4))
plt.hist(dif, bins=12, color="slateblue", alpha=0.7)
plt.title("Diferencias pareadas (Wilcoxon asume simetría)")
plt.tight_layout(); plt.show()

## 3. Robustez frente a outliers

Un test de rangos casi no se inmuta ante 3 outliers extremos; el t-test se distorsiona porque usa las medias.

In [ ]:
base = rng.normal(50, 5, 100)
contaminado = base.copy()
contaminado[:3] = 200.0     # 3 outliers
otro = rng.normal(52, 5, 100)
t_p = stats.ttest_ind(contaminado, otro, equal_var=False).pvalue
mw_p = stats.mannwhitneyu(contaminado, otro).pvalue
print(f"medias: contaminado={contaminado.mean():.1f}  otro={otro.mean():.1f}")
print(f"Welch t p={t_p:.3f}   Mann-Whitney p={mw_p:.4f}")
print("Los outliers inflan la media y rompen el t-test; los rangos son robustos.")

## 4. Kruskal-Wallis (≥ 3 grupos)

Extiende Mann-Whitney a `k` grupos. Bajo `H₀`, `H ~ χ²(k-1)`.

In [ ]:
g1 = rng.lognormal(1.0, 0.4, 60)
g2 = rng.lognormal(1.2, 0.4, 60)
g3 = rng.lognormal(1.5, 0.4, 60)
H, p = stats.kruskal(g1, g2, g3)
print(f"Kruskal-Wallis H={H:.2f}  p={p:.2e}")
assert p < 0.01

## 5. Post-hoc por pares con corrección Holm

Kruskal solo dice que *al menos uno* difiere. Hacemos Mann-Whitney por pares y corregimos con Holm (alternativa a Dunn cuando no hay `scikit-posthocs`).

In [ ]:
pares = [("g1", "g2", g1, g2), ("g1", "g3", g1, g3), ("g2", "g3", g2, g3)]
raw = [stats.mannwhitneyu(x, y).pvalue for _, _, x, y in pares]
rej, adj, _, _ = multipletests(raw, alpha=0.05, method="holm")
for (n1, n2, _, _), pr, pa, r in zip(pares, raw, adj, rej):
    print(f"{n1} vs {n2}: p={pr:.4f}  p_holm={pa:.4f}  {'sig' if r else 'ns'}")
assert rej.sum() >= 1
print("\nReportá mediana ± IQR (no media ± SD) con datos asimétricos:")
for name, g in (("g1", g1), ("g2", g2), ("g3", g3)):
    q1, q3 = np.percentile(g, [25, 75])
    print(f"  {name}: mediana={np.median(g):.2f}  IQR=({q1:.2f}, {q3:.2f})")

## Ejercicios

1. Compará el p de Mann-Whitney del bloque 1 contra un Welch t-test sobre los mismos datos lognormales.
2. Calculá Cliff's δ para los grupos `a` y `b` y verificá que concuerda cualitativamente con la rank-biserial.
3. Con datos razonablemente simétricos y `n=200`, mostrá que el t-test tiene un poco más de poder que Mann-Whitney (eficiencia ≈ 0.955).

## Conclusiones

- Los no paramétricos testean superioridad estocástica / distribuciones, no medias: reportá **medianas**, no promedios.
- Mann-Whitney ↔ dos grupos, Wilcoxon ↔ pareado, Kruskal-Wallis ↔ ≥ 3 grupos.
- Son robustos a outliers y válidos para datos ordinales (Likert).
- Tras Kruskal, hacé post-hoc con corrección por múltiples comparaciones (Dunn o Mann-Whitney + Holm).

## ✅ Soluciones de los ejercicios
<!--SOL175184-->

Soluciones trabajadas y **ejecutables** de todos los ejercicios de la sección *🧪 Ejercicios*. Datos sintéticos reproducibles con `np.random.default_rng(42)`; sin dependencias de internet. Cada bloque incluye `assert`/`print` para autocorregir.

<!--SOL175184-->

**Ej. 1 — Mann-Whitney U** de `tip` por `sex` + rank-biserial r.

In [ ]:
# <!--SOL175184-->
import numpy as np
from scipy import stats
rng = np.random.default_rng(42)
n = 244
sex = rng.choice(["Male", "Female"], size=n, p=[0.64, 0.36])
tip = np.clip(rng.lognormal(2.85, 0.38, n)*0.15 + rng.normal(0, 0.5, n) + np.where(sex=="Male", 0.15, 0.0), 1.0, None)
a = tip[sex == "Male"]; b = tip[sex == "Female"]
U, p = stats.mannwhitneyu(a, b, alternative="two-sided")
r_rb = 1 - 2*U/(len(a)*len(b))
print(f"U={U:.1f} p={p:.4f} rank-biserial r={r_rb:.3f}")
assert -1 <= r_rb <= 1
print("Mann-Whitney + rank-biserial: OK")

<!--SOL175184-->

**Ej. 2 — Wilcoxon signed-rank** (pareado) + simetría de diferencias.

In [ ]:
# <!--SOL175184-->
import matplotlib.pyplot as plt
rng0 = np.random.default_rng(0)
antes = rng0.normal(140, 12, 30); despues = antes - rng0.normal(5, 3, 30)
w = stats.wilcoxon(antes, despues); diffs = antes - despues
print(f"Wilcoxon stat={w.statistic:.1f} p={w.pvalue:.2e} skew(diffs)={stats.skew(diffs):.2f}")
fig, ax = plt.subplots(figsize=(5, 3)); ax.hist(diffs, bins=12); ax.set_title("Diferencias antes-despues")
assert w.pvalue < 0.05
print("Wilcoxon detecta el efecto: OK")

<!--SOL175184-->

**Ej. 3 — Robustez a outliers:** Welch's t se rompe, Mann-Whitney no.

In [ ]:
# <!--SOL175184-->
g1 = rng.normal(50, 5, 100)
g1c = np.concatenate([g1, [200, 200, 200]])
g2 = rng.normal(52, 5, 103)
p_t = stats.ttest_ind(g1c, g2, equal_var=False).pvalue
p_mw = stats.mannwhitneyu(g1c, g2, alternative="two-sided").pvalue
print(f"Welch p={p_t:.3f}  Mann-Whitney p={p_mw:.4f}")
assert p_mw < p_t
print("Mann-Whitney mas robusto a outliers: OK")

<!--SOL175184-->

**Ej. 4 — Kruskal-Wallis** (≥3 grupos) vs ANOVA.

In [ ]:
# <!--SOL175184-->
specs = {"Adelie": (3700, 460, 152), "Gentoo": (5080, 500, 124), "Chinstrap": (3733, 384, 68)}
grupos = [rng.normal(mu, sd, k) for mu, sd, k in specs.values()]
H, pk = stats.kruskal(*grupos); F, pf = stats.f_oneway(*grupos)
print(f"Kruskal H={H:.2f} p={pk:.2e}   ANOVA F={F:.2f} p={pf:.2e}")
assert pk < 0.05
print("Kruskal-Wallis concuerda con ANOVA: OK")

<!--SOL175184-->

**Ej. 5 — Post-hoc Dunn + Holm** (implementado a mano, sin `scikit_posthocs`).

In [ ]:
# <!--SOL175184-->
from statsmodels.stats.multitest import multipletests
def dunn_test(groups):
    allv = np.concatenate(groups); ranks = stats.rankdata(allv); N = len(allv)
    _, cnt = np.unique(allv, return_counts=True)
    tie = (cnt**3 - cnt).sum()
    sigma2 = (N*(N+1)/12) - tie/(12*(N-1))
    idx = np.cumsum([0] + [len(g) for g in groups])
    mean_rank = [ranks[idx[i]:idx[i+1]].mean() for i in range(len(groups))]
    ni = [len(g) for g in groups]; pairs, pvals = [], []
    for i in range(len(groups)):
        for j in range(i+1, len(groups)):
            z = (mean_rank[i] - mean_rank[j]) / np.sqrt(sigma2*(1/ni[i] + 1/ni[j]))
            pairs.append((i, j)); pvals.append(2*stats.norm.sf(abs(z)))
    rej, adj, _, _ = multipletests(pvals, method="holm")
    return pairs, adj, rej
pairs, padj, rej = dunn_test(grupos)
for (i, j), pa, r in zip(pairs, padj, rej):
    print(f"g{i} vs g{j}: p_holm={pa:.3e} {'*' if r else ''}")
assert rej.any()
print("Dunn + Holm: OK")